**Frozen BERT phrases → recipe attention → UMAP → HDBSCAN → denoising autoencoder**

Keep individual phrases from `products` (or `product`), `adj`, and `verb`.
Cache one frozen `bert-base-uncased` vector per phrase. Separate self-attention
contextualizes ingredients, adjectives, and verbs within each recipe; ingredients
then cross-attend to that recipe's contextualized adjectives and verbs.

Pretrain the attention encoder by reconstructing masked phrase embeddings using
content only. Freeze its best validation checkpoint before generating features.
Mean-pool and unit-normalize each contextualized field, then combine with fixed
weights **0.6 × products + 0.2 × adj + 0.2 × verb**. Empty fields contribute zero;
missing columns raise an error. No positional embeddings are used.

Both datasets share this encoder, one UMAP projection, and one HDBSCAN clustering.
Cluster means and the recommender use the same 768-dimensional contextualized
vectors. Ratings, user IDs, calories, and other numeric fields never enter the
content encoder or clustering. Retrain the recommender after changing the encoder;
old recommender weights are not compatible with the new feature space.

Run the installation cell before imports, then restart the kernel.
The first BERT load downloads pretrained weights from Hugging Face.

In [ ]:
%pip install transformers==5.17.0 numpy pandas 'scikit-learn>=1.3' scipy umap-learn ipython torch torchvision torchaudio h5py

**After installing: restart the kernel, then run the cells below.**

Use **Kernel → Restart Kernel** (or your editor's Restart command). This clears
Transformers' cached backend detection and previously imported package versions.
The verification cell prints the active Python interpreter and library versions.
If installation fails, fix that error before continuing.

The pip package is `torch`. For a specific CUDA/ROCm build, use the command matching
your hardware from the [official PyTorch installer](https://pytorch.org/get-started/locally/)
in this notebook's kernel before restarting.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import normalize
from transformers import AutoModel, AutoTokenizer
from umap import UMAP

**Model definitions: recipe attention encoder and cluster recommender**

Run the following cell once before the remaining cells. It contains both models,
their data helpers, training, checkpoint loading, and prediction code. No local
`recipe_encoder.py` or `cluster_recommender.py` import is required.


In [ ]:
# Recipe attention encoder and cluster recommender — self-contained definitions
# Recipe attention encoder

"""Recipe-local attention, pretrained on masked frozen BERT phrase embeddings.

The encoder is frozen before clustering. Ratings and user IDs never enter this
module, and repeated rating rows do not increase a recipe's pretraining weight.
"""

import ast
from dataclasses import dataclass
from functools import lru_cache
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset


FEATURE_COLUMNS = ("products", "adj", "verb")
COLUMN_WEIGHTS = {"products": 0.6, "adj": 0.2, "verb": 0.2}
FEATURE_FORMAT_VERSION = "recipe_context_v1"


@lru_cache(maxsize=200_000)
def _string_terms(value):
    value = value.strip()
    if value.lower() in {"", "nan", "none", "null", "<na>"}:
        return ()
    if value.startswith(("[", "(")):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError) as exc:
                raise ValueError(f"Invalid term list: {value[:120]!r}") from exc
        return feature_to_terms(parsed)
    return (" ".join(value.lower().split()),)


def feature_to_terms(value):
    """Flatten nested lists without splitting multiword phrases or dropping repeats."""
    if isinstance(value, str):
        return _string_terms(value)
    if isinstance(value, (list, tuple, np.ndarray)):
        return tuple(term for item in value for term in feature_to_terms(item))
    if value is None or pd.isna(value):
        return ()
    raise TypeError(f"Expected text or a list of terms, got {type(value).__name__}.")


@dataclass
class TermFeatureTable:
    unique_features: pd.DataFrame
    row_embedding_ids: np.ndarray
    row_counts: np.ndarray
    terms: tuple
    term_ids: dict


def prepare_term_features(datasets):
    """Canonicalize complete feature combinations, retaining positional row IDs.

    ``datasets`` is an ordered mapping of names to dataframes. Only the three
    content fields are read; original frames and their indexes are not modified.
    Term IDs refer to a shared phrase vocabulary, not recipe or user IDs.
    """
    total = sum(len(frame) for frame in datasets.values())
    if not total:
        raise ValueError("Both datasets are empty; there are no rows to encode.")
    row_ids = np.empty(total, dtype=np.int64)
    lookup, keys = {}, []
    offset = 0
    for name, frame in datasets.items():
        product_column = "products" if "products" in frame.columns else "product"
        columns = (product_column, "adj", "verb")
        missing = [column for column in columns if column not in frame.columns]
        if missing:
            raise ValueError(f"Missing embedding columns in {name}: {missing}")
        for values in frame[list(columns)].itertuples(index=False, name=None):
            key = tuple(tuple(sorted(feature_to_terms(value))) for value in values)
            if key not in lookup:
                lookup[key] = len(keys)
                keys.append(key)
            row_ids[offset] = lookup[key]
            offset += 1
    _string_terms.cache_clear()
    vocabulary = {}
    term_ids = {field: [] for field in FEATURE_COLUMNS}
    for key in keys:
        for field, phrases in zip(FEATURE_COLUMNS, key):
            ids = []
            for phrase in phrases:
                if phrase not in vocabulary:
                    vocabulary[phrase] = len(vocabulary)
                ids.append(vocabulary[phrase])
            term_ids[field].append(np.asarray(ids, dtype=np.int64))
    return TermFeatureTable(
        pd.DataFrame(keys, columns=FEATURE_COLUMNS), row_ids,
        np.bincount(row_ids, minlength=len(keys)), tuple(vocabulary), term_ids,
    )


def embed_terms(terms, tokenizer, bert_model, device, batch_size=32, max_length=512):
    """Encode each cached phrase once; freeze BERT and exclude special/pad tokens."""
    if batch_size < 1 or max_length < 3:
        raise ValueError("Use a positive batch size and max_length >= 3.")
    bert_model.to(device).eval().requires_grad_(False)
    embeddings = np.zeros((len(terms), bert_model.config.hidden_size), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(terms), batch_size):
            inputs = tokenizer(
                list(terms[start:start + batch_size]), padding=True, truncation=True,
                max_length=max_length, return_special_tokens_mask=True, return_tensors="pt",
            )
            special = inputs.pop("special_tokens_mask").to(device).bool()
            inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
            hidden = bert_model(**inputs).last_hidden_state.float()
            valid = (inputs["attention_mask"].bool() & ~special).unsqueeze(-1)
            pooled = (hidden * valid).sum(1) / valid.sum(1).clamp(min=1)
            embeddings[start:start + len(pooled)] = F.normalize(pooled, dim=-1).cpu().numpy()
    return embeddings


class _AttentionBlock(nn.Module):
    """Post-norm attention/FFN block, bypassing rows without queries or keys."""

    def __init__(self, dim, heads, feedforward_dim, dropout):
        super().__init__()
        self.attention = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.attention_norm = nn.LayerNorm(dim)
        self.feedforward_norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.feedforward = nn.Sequential(
            nn.Linear(dim, feedforward_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(feedforward_dim, dim),
        )

    def forward(self, queries, memory, query_mask, memory_mask):
        # Never send an all-masked key row to softmax (which can produce NaNs).
        active = (query_mask.any(1) & memory_mask.any(1)).nonzero(as_tuple=True)[0]
        result = queries.clone()
        if active.numel():
            q = queries[active]
            kv = memory[active]
            attended, _ = self.attention(
                q, kv, kv, key_padding_mask=~memory_mask[active], need_weights=False,
            )
            x = self.attention_norm(q + self.dropout(attended))
            x = self.feedforward_norm(x + self.dropout(self.feedforward(x)))
            result = result.index_copy(0, active, x)
        return result.masked_fill(~query_mask.unsqueeze(-1), 0.0)


class RecipeContextEncoder(nn.Module):
    """Three self-attention streams followed by ingredients -> adj/verb attention.

    Inputs have shape [recipes, terms, embedding_dim]; validity masks are bool
    [recipes, terms], with True for real terms. Zero-width fields are supported.
    ``masked_positions`` optionally replaces selected real inputs with learned
    mask vectors BEFORE any attention, so reconstruction cannot see its targets.

    Returns dictionaries ``term_embeddings`` and ``field_embeddings`` plus a
    pooled ``recipe_embedding``. No positional encoding is used.
    """

    def __init__(self, embedding_dim=768, num_heads=8, feedforward_dim=1536,
                 dropout=0.1, column_weights=None):
        super().__init__()
        if embedding_dim < 1 or num_heads < 1 or embedding_dim % num_heads:
            raise ValueError("embedding_dim must be positive and divisible by num_heads.")
        if feedforward_dim < 1 or not 0 <= dropout < 1:
            raise ValueError("Use positive feedforward_dim and 0 <= dropout < 1.")
        weights = dict(COLUMN_WEIGHTS if column_weights is None else column_weights)
        values = np.asarray([weights[field] for field in FEATURE_COLUMNS], dtype=np.float32)
        if not (np.isfinite(values).all() and values[0] > values[1] > 0
                and values[1] == values[2] and np.isclose(values.sum(), 1.0)):
            raise ValueError("Use positive weights summing to 1 with products > adj = verb.")
        self.config = dict(embedding_dim=embedding_dim, num_heads=num_heads,
                           feedforward_dim=feedforward_dim, dropout=dropout, column_weights=weights)
        self.register_buffer("column_weights", torch.from_numpy(values))
        self.field_types = nn.Parameter(torch.empty(3, embedding_dim))
        self.mask_tokens = nn.Parameter(torch.empty(3, embedding_dim))
        nn.init.normal_(self.field_types, std=0.02)
        nn.init.normal_(self.mask_tokens, std=0.02)
        self.self_attention = nn.ModuleDict({
            field: _AttentionBlock(embedding_dim, num_heads, feedforward_dim, dropout)
            for field in FEATURE_COLUMNS
        })
        self.cross_attention = _AttentionBlock(embedding_dim, num_heads, feedforward_dim, dropout)
        self.reconstruction_heads = nn.ModuleDict({
            field: nn.Linear(embedding_dim, embedding_dim) for field in FEATURE_COLUMNS
        })

    def forward(self, products, adj, verb, products_mask, adj_mask, verb_mask,
                masked_positions=None):
        features = dict(zip(FEATURE_COLUMNS, (products, adj, verb)))
        masks = dict(zip(FEATURE_COLUMNS, (products_mask, adj_mask, verb_mask)))
        contextual = {}
        for index, field in enumerate(FEATURE_COLUMNS):
            x, valid = features[field], masks[field]
            if (x.ndim != 3 or x.shape[0] != products.shape[0]
                    or x.shape[-1] != self.config["embedding_dim"]
                    or valid.shape != x.shape[:2] or valid.dtype != torch.bool):
                raise ValueError(f"Invalid embedding or bool validity-mask shape for {field}.")
            if masked_positions is not None:
                hidden = masked_positions[field]
                if hidden.shape != valid.shape or hidden.dtype != torch.bool:
                    raise ValueError(f"Invalid reconstruction mask for {field}.")
                x = torch.where(hidden.unsqueeze(-1), self.mask_tokens[index], x)
            x = (x + self.field_types[index]).masked_fill(~valid.unsqueeze(-1), 0.0)
            contextual[field] = self.self_attention[field](x, x, valid, valid)
        memory = torch.cat([contextual["adj"], contextual["verb"]], dim=1)
        memory_mask = torch.cat([adj_mask, verb_mask], dim=1)
        contextual["products"] = self.cross_attention(
            contextual["products"], memory, products_mask, memory_mask,
        )
        pooled = {}
        for field in FEATURE_COLUMNS:
            mean = contextual[field].sum(1) / masks[field].sum(1, keepdim=True).clamp(min=1)
            pooled[field] = F.normalize(mean, dim=-1)
        recipe = sum(self.column_weights[i] * pooled[field] for i, field in enumerate(FEATURE_COLUMNS))
        return {"term_embeddings": contextual, "field_embeddings": pooled, "recipe_embedding": recipe}


def eligible_mask_targets(term_ids, feature_id):
    """Only hide terms with another visible term reachable by their attention."""
    lengths = [len(term_ids[field][feature_id]) for field in FEATURE_COLUMNS]
    targets = []
    for field, length in enumerate(lengths):
        reachable = sum(lengths) if field == 0 else length
        if reachable > 1:
            targets.extend((field, position) for position in range(length))
    return targets


def split_term_features(term_ids, validation_fraction=0.1, seed=42):
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1.")
    eligible = np.asarray([
        index for index in range(len(term_ids["products"]))
        if eligible_mask_targets(term_ids, index)
    ], dtype=np.int64)
    if not len(eligible):
        raise ValueError("No feature combination has a term with visible attention context for pretraining.")
    order = np.random.default_rng(seed).permutation(eligible)
    n_val = min(max(1, round(len(order) * validation_fraction)), len(order) - 1)
    return order[n_val:], order[:n_val]


class MaskedTermDataset(Dataset):
    def __init__(self, term_ids, feature_ids, seed=42, fixed=False):
        self.term_ids, self.feature_ids = term_ids, np.asarray(feature_ids, dtype=np.int64)
        self.rng = np.random.default_rng(seed)
        self.fixed_targets = [self._choose(index) for index in self.feature_ids] if fixed else None

    def _choose(self, feature_id):
        candidates = eligible_mask_targets(self.term_ids, feature_id)
        if not candidates:
            raise ValueError(f"Feature {feature_id} has no eligible masked term.")
        return candidates[self.rng.integers(len(candidates))]

    def __len__(self):
        return len(self.feature_ids)

    def __getitem__(self, index):
        feature_id = self.feature_ids[index]
        field, position = self.fixed_targets[index] if self.fixed_targets is not None else self._choose(feature_id)
        return int(feature_id), field, position


class TermBatchCollator:
    """Gather cached BERT vectors only for this batch; pad fields independently."""

    def __init__(self, term_embeddings, term_ids):
        self.embeddings = np.asarray(term_embeddings, dtype=np.float32)
        self.term_ids = term_ids

    def collate(self, feature_ids, targets=None):
        batch_size, dim = len(feature_ids), self.embeddings.shape[1]
        inputs, hidden = {}, {}
        for field_index, field in enumerate(FEATURE_COLUMNS):
            rows = [self.term_ids[field][index] for index in feature_ids]
            width = max((len(row) for row in rows), default=0)
            values = np.zeros((batch_size, width, dim), dtype=np.float32)
            valid = np.zeros((batch_size, width), dtype=bool)
            masked = np.zeros_like(valid)
            for owner, ids in enumerate(rows):
                values[owner, :len(ids)] = self.embeddings[ids]
                valid[owner, :len(ids)] = True
                if targets is not None and targets[owner][0] == field_index:
                    masked[owner, targets[owner][1]] = True
            inputs[field] = torch.from_numpy(values)
            inputs[f"{field}_mask"] = torch.from_numpy(valid)
            hidden[field] = torch.from_numpy(masked)
        if targets is None:
            return {"inputs": inputs}
        inputs["masked_positions"] = hidden
        target_ids = [self.term_ids[FEATURE_COLUMNS[field]][index][position]
                      for index, (field, position) in zip(feature_ids, targets)]
        return {
            "inputs": inputs,
            "target_fields": torch.tensor([field for field, _ in targets], dtype=torch.long),
            "target_positions": torch.tensor([position for _, position in targets], dtype=torch.long),
            "target_embeddings": torch.from_numpy(self.embeddings[target_ids]),
        }

    def __call__(self, samples):
        return self.collate([index for index, _, _ in samples],
                            [(field, position) for _, field, position in samples])


def _to_device(value, device):
    if isinstance(value, dict):
        return {key: _to_device(item, device) for key, item in value.items()}
    return value.to(device)


def masked_term_loss(model, batch):
    result = model(**batch["inputs"])
    predictions = torch.zeros_like(batch["target_embeddings"])
    for index, field in enumerate(FEATURE_COLUMNS):
        owners = (batch["target_fields"] == index).nonzero(as_tuple=True)[0]
        if owners.numel():
            terms = result["term_embeddings"][field][owners, batch["target_positions"][owners]]
            predictions = predictions.index_copy(0, owners, model.reconstruction_heads[field](terms))
    return (1 - F.cosine_similarity(predictions, batch["target_embeddings"], dim=-1)).mean()


def train_recipe_encoder(model, term_embeddings, term_ids, device, epochs=10,
                         batch_size=64, learning_rate=1e-4, validation_fraction=0.1, seed=42):
    """Pretrain, restore the best validation checkpoint, then freeze the encoder."""
    if epochs < 1 or batch_size < 1 or not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError("Epochs, batch size, and learning rate must be positive.")
    train_ids, validation_ids = split_term_features(term_ids, validation_fraction, seed)
    collator = TermBatchCollator(term_embeddings, term_ids)
    train_data = MaskedTermDataset(term_ids, train_ids, seed)
    validation_data = MaskedTermDataset(term_ids, validation_ids, seed + 1, fixed=True)
    loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0,
                        collate_fn=collator, generator=torch.Generator().manual_seed(seed))
    validation_loader = DataLoader(validation_data, batch_size=batch_size, num_workers=0,
                                   collate_fn=collator)
    torch.manual_seed(seed)
    model.to(device).requires_grad_(True)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_state, best_loss = [], None, np.inf
    for epoch in range(1, epochs + 1):
        model.train()
        training_total = 0.0
        for batch in loader:
            batch = _to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)
            loss = masked_term_loss(model, batch)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite masked-term training loss.")
            loss.backward()
            optimizer.step()
            training_total += loss.detach().item() * len(batch["target_fields"])
        model.eval()
        validation_total = 0.0
        with torch.inference_mode():
            for batch in validation_loader:
                batch = _to_device(batch, device)
                loss = masked_term_loss(model, batch)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Non-finite masked-term validation loss.")
                validation_total += loss.item() * len(batch["target_fields"])
        validation_loss = validation_total / len(validation_data) if len(validation_data) else np.nan
        history.append({"epoch": epoch, "train_cosine_loss": training_total / len(train_data),
                        "validation_cosine_loss": validation_loss,
                        "training_features": len(train_data), "validation_features": len(validation_data)})
        print(history[-1], flush=True)
        if len(validation_data) and validation_loss < best_loss:
            best_loss = validation_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    model.zero_grad(set_to_none=True)
    model.eval().requires_grad_(False)
    return pd.DataFrame(history)


def encode_feature_table(model, term_embeddings, term_ids, device, batch_size=64):
    """Infer all unique combinations, storing only pooled vectors on the CPU."""
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    model.to(device).eval()
    count, dim = len(term_ids["products"]), model.config["embedding_dim"]
    fields = {field: np.zeros((count, dim), dtype=np.float32) for field in FEATURE_COLUMNS}
    recipes = np.zeros((count, dim), dtype=np.float32)
    collator = TermBatchCollator(term_embeddings, term_ids)
    with torch.inference_mode():
        for start in range(0, count, batch_size):
            stop = min(start + batch_size, count)
            batch = _to_device(collator.collate(range(start, stop)), device)
            result = model(**batch["inputs"])
            for field in FEATURE_COLUMNS:
                fields[field][start:stop] = result["field_embeddings"][field].cpu().numpy()
            recipes[start:stop] = result["recipe_embedding"].cpu().numpy()
    if not np.isfinite(recipes).all():
        raise FloatingPointError("Encoder produced non-finite recipe embeddings.")
    return fields, recipes


def encoder_identity(model, bert_model_name, max_length):
    """Fingerprint both feature semantics and weights to pair encoder/recommender."""
    metadata = {"feature_format_version": FEATURE_FORMAT_VERSION, "config": model.config,
                "bert_model_name": bert_model_name, "max_length": max_length}
    digest = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode())
    for name, tensor in sorted(model.state_dict().items()):
        array = tensor.detach().cpu().contiguous().numpy()
        digest.update(name.encode())
        digest.update(str(array.dtype).encode())
        digest.update(str(array.shape).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()


def save_recipe_encoder(model, path, bert_model_name="bert-base-uncased", max_length=512,
                        training_config=None):
    import h5py

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    identity = encoder_identity(model, bert_model_name, max_length)
    with h5py.File(path, "w") as handle:
        handle.attrs.update({
            "format": "pytorch_state_dict", "model_class": "RecipeContextEncoder",
            "feature_format_version": FEATURE_FORMAT_VERSION, "encoder_identity": identity,
            "config": json.dumps(model.config, sort_keys=True), "bert_model_name": bert_model_name,
            "max_length": max_length, "training_config": json.dumps(training_config or {}, sort_keys=True),
        })
        weights = handle.create_group("state_dict")
        for name, tensor in model.state_dict().items():
            weights.create_dataset(name, data=tensor.detach().cpu().numpy(), compression="gzip")
    return identity


def load_recipe_encoder(path, device="cpu"):
    """Restore a frozen encoder and metadata; reject incompatible/corrupted files."""
    import h5py

    with h5py.File(path, "r") as handle:
        metadata = dict(handle.attrs)
        if (metadata.get("feature_format_version") != FEATURE_FORMAT_VERSION
                or metadata.get("model_class") != "RecipeContextEncoder"):
            raise ValueError("Incompatible recipe encoder feature format or model class.")
        metadata["config"] = json.loads(metadata["config"])
        metadata["training_config"] = json.loads(metadata["training_config"])
        model = RecipeContextEncoder(**metadata["config"])
        model.load_state_dict({name: torch.from_numpy(value[...])
                               for name, value in handle["state_dict"].items()})
    actual = encoder_identity(model, metadata["bert_model_name"], int(metadata["max_length"]))
    if actual != metadata["encoder_identity"]:
        raise ValueError("Encoder identity does not match its configuration and weights.")
    model.to(device).eval().requires_grad_(False)
    return model, metadata


# Cluster-conditioned denoising autoencoder and recommendation

from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset


@dataclass
class RatingGroup:
    user_id: str
    cluster: int
    items: np.ndarray
    ratings: np.ndarray


def downsample_dominant_user_ratings(
    recipe_df, dominant_rating=5.0, heavy_user_min_ratings=50,
    max_dominant_to_other_ratio=2.0, min_dominant_keep=10,
    zero_is_missing=True, seed=42,
):
    """Remove excess dominant ratings only from prolific users.

    For a user with at least ``heavy_user_min_ratings`` observed ratings, retain
    at most max(``min_dominant_keep``, ceil(ratio * non-dominant ratings))
    occurrences of ``dominant_rating``. Other ratings and less-active users are
    unchanged. Selection is deterministic for a fixed seed.
    """
    required = {"user_id", "recipe_id", "rating"}
    missing = required.difference(recipe_df.columns)
    if missing:
        raise ValueError(f"Missing rating columns: {sorted(missing)}")
    if heavy_user_min_ratings < 1 or min_dominant_keep < 1:
        raise ValueError("User threshold and minimum retained ratings must be positive.")
    if not np.isfinite(max_dominant_to_other_ratio) or max_dominant_to_other_ratio < 0:
        raise ValueError("max_dominant_to_other_ratio must be finite and nonnegative.")

    ratings = pd.to_numeric(recipe_df["rating"], errors="coerce")
    observed = ratings.notna() & recipe_df["user_id"].notna()
    if zero_is_missing:
        observed &= ratings.ne(0)
    work = pd.DataFrame({
        "position": np.arange(len(recipe_df), dtype=np.int64),
        "user_id": recipe_df["user_id"].astype(str),
        "rating": ratings,
    }).loc[observed]
    user_total = work.groupby("user_id", sort=False).size().rename("total")
    user_dominant = (
        work.loc[work["rating"].eq(dominant_rating)]
        .groupby("user_id", sort=False).size().rename("dominant")
        .reindex(user_total.index, fill_value=0)
    )
    user_other = user_total - user_dominant
    heavy_users = user_total.index[user_total.ge(heavy_user_min_ratings)]
    dominant_limits = np.maximum(
        min_dominant_keep,
        np.ceil(max_dominant_to_other_ratio * user_other).astype(np.int64),
    )
    keep = np.ones(len(recipe_df), dtype=bool)
    rng = np.random.default_rng(seed)
    heavy_dominant = work.loc[
        work["user_id"].isin(heavy_users) & work["rating"].eq(dominant_rating)
    ]
    for user_id, rows in heavy_dominant.groupby("user_id", sort=False):
        positions = rows["position"].to_numpy(dtype=np.int64)
        limit = int(dominant_limits.loc[user_id])
        if len(positions) > limit:
            keep[rng.choice(positions, size=len(positions) - limit, replace=False)] = False

    balanced = recipe_df.iloc[np.flatnonzero(keep)].copy()
    before = ratings.loc[observed].value_counts().sort_index()
    after_values = pd.to_numeric(balanced["rating"], errors="coerce")
    after_observed = after_values.notna()
    if zero_is_missing:
        after_observed &= after_values.ne(0)
    after = after_values.loc[after_observed].value_counts().sort_index()
    levels = before.index.union(after.index).sort_values()
    summary = pd.DataFrame({
        "before": before.reindex(levels, fill_value=0).astype(np.int64),
        "after": after.reindex(levels, fill_value=0).astype(np.int64),
    })
    summary["removed"] = summary["before"] - summary["after"]
    summary["before_percent"] = 100 * summary["before"] / summary["before"].sum()
    summary["after_percent"] = 100 * summary["after"] / summary["after"].sum()
    summary.index.name = "rating"
    summary.attrs["heavy_users"] = len(heavy_users)
    summary.attrs["removed_rows"] = int((~keep).sum())
    return balanced, summary


def build_catalog_and_ratings(recipe_df, food_df, rating_min=1.0, rating_max=5.0,
                              zero_is_missing=True, rating_df=None):
    """Build the full catalog while optionally taking ratings from a balanced subset."""
    if rating_max <= rating_min:
        raise ValueError("rating_max must be greater than rating_min.")
    catalogs = []
    for frame, source, id_column, title_column in (
        (recipe_df, "recipe", "recipe_id", "name"),
        (food_df, "food", "fdc_id", "description"),
    ):
        if frame[id_column].isna().any():
            raise ValueError(f"Missing item IDs in {source} data.")
        items = pd.DataFrame({
            "item_key": source + ":" + frame[id_column].astype(str),
            "source": source,
            "title": frame[title_column].fillna("").astype(str),
            "embedding_row_id": frame["embedding_row_id"],
            "cluster": frame["cluster"],
        })
        inconsistent = items.groupby("item_key")[["embedding_row_id", "cluster"]].nunique().gt(1).any(axis=1)
        if inconsistent.any():
            raise ValueError(f"An item has inconsistent features/clusters: {inconsistent[inconsistent].index[0]}")
        catalogs.append(items.drop_duplicates("item_key"))
    catalog = pd.concat(catalogs, ignore_index=True)
    catalog.index.name = "item_index"

    rating_source = recipe_df if rating_df is None else rating_df
    raw = rating_source[["user_id", "recipe_id", "rating"]].copy()
    raw["rating"] = pd.to_numeric(raw["rating"], errors="coerce")
    raw = raw.loc[raw["user_id"].notna() & raw["recipe_id"].notna() & raw["rating"].notna()]
    if zero_is_missing:
        raw = raw.loc[raw["rating"] != 0]
    if not raw["rating"].between(rating_min, rating_max).all():
        raise ValueError("Observed ratings lie outside the configured rating scale.")
    raw["user_id"] = raw["user_id"].astype(str)
    raw["item_key"] = "recipe:" + raw["recipe_id"].astype(str)
    ratings = raw.groupby(["user_id", "item_key"], as_index=False, sort=False)["rating"].mean()
    ratings = ratings.merge(catalog.reset_index()[["item_key", "item_index", "cluster"]],
                            on="item_key", how="left", validate="many_to_one")
    if ratings["item_index"].isna().any():
        raise ValueError("Some rated items are missing from the catalog.")
    ratings["scaled_rating"] = (ratings["rating"] - rating_min) / (rating_max - rating_min)
    return catalog, ratings


def split_rating_groups(ratings, validation_fraction=0.2, seed=42):
    """Hold out fixed targets first. At least two training observations remain.

    Training uses groups with >=2 observations. Singleton groups remain available
    for recommendation context but cannot support masked reconstruction with a
    nonempty context. HDBSCAN noise is never treated as a shared cluster.
    """
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1.")
    rng = np.random.default_rng(seed)
    training, validation = [], []
    eligible = ratings.loc[ratings["cluster"] >= 0]
    for (user, cluster), group in eligible.groupby(["user_id", "cluster"], sort=False):
        items = group["item_index"].to_numpy(dtype=np.int64)
        values = group["scaled_rating"].to_numpy(dtype=np.float32)
        if len(items) < 2:
            continue
        order = rng.permutation(len(items))
        n_val = min(max(1, round(len(items) * validation_fraction)), len(items) - 2) if len(items) >= 3 else 0
        val_ids, train_ids = order[:n_val], order[n_val:]
        train_group = RatingGroup(str(user), int(cluster), items[train_ids], values[train_ids])
        training.append(train_group)
        if n_val:
            targets = RatingGroup(str(user), int(cluster), items[val_ids], values[val_ids])
            validation.append((train_group, targets))
    if not training:
        raise ValueError("No user has two observed ratings in the same non-noise cluster. Inspect clustering and rating coverage.")
    return training, validation


class MaskedRatingDataset(Dataset):
    """Resample masked observed targets every access; targets never enter context."""

    def __init__(self, groups, mask_fraction=0.3, max_context=64, max_targets=32, seed=42):
        if not 0 < mask_fraction < 1 or max_context < 1 or max_targets < 1:
            raise ValueError("Use 0 < mask_fraction < 1 and positive context/target limits.")
        self.groups = groups
        self.mask_fraction = mask_fraction
        self.max_context, self.max_targets = max_context, max_targets
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, index):
        group = self.groups[index]
        order = self.rng.permutation(len(group.items))
        n_mask = min(len(order) - 1, max(1, round(len(order) * self.mask_fraction)))
        targets = order[:n_mask][:self.max_targets]
        context = order[n_mask:][:self.max_context]
        return (group.cluster, group.items[context], group.ratings[context],
                group.items[targets], group.ratings[targets])


class ValidationRatingDataset(Dataset):
    """Fixed holdouts; context contains only the training part of each group."""

    def __init__(self, pairs, max_context=64, seed=42):
        rng = np.random.default_rng(seed)
        self.samples = []
        for context, target in pairs:
            chosen = rng.permutation(len(context.items))[:max_context]
            self.samples.append((context.cluster, context.items[chosen], context.ratings[chosen],
                                 target.items, target.ratings))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


class RatingCollator:
    """Gather only the embeddings needed by this minibatch from the shared table."""

    def __init__(self, feature_matrix, catalog, cluster_embeddings):
        self.features = feature_matrix
        self.embedding_ids = catalog["embedding_row_id"].to_numpy(dtype=np.int64)
        self.centroids = cluster_embeddings["average_embedding"].to_dict()

    def __call__(self, samples):
        width = max(len(sample[1]) for sample in samples)
        context_ids = np.zeros((len(samples), width), dtype=np.int64)
        context_ratings = np.zeros((len(samples), width), dtype=np.float32)
        context_mask = np.zeros((len(samples), width), dtype=bool)
        target_ids, target_ratings, owners, cluster_features = [], [], [], []
        for owner, (cluster, items, ratings, targets, values) in enumerate(samples):
            length = len(items)
            context_ids[owner, :length] = self.embedding_ids[items]
            context_ratings[owner, :length] = ratings
            context_mask[owner, :length] = True
            target_ids.extend(self.embedding_ids[targets])
            target_ratings.extend(values)
            owners.extend([owner] * len(targets))
            cluster_features.append(self.centroids[cluster])
        return {
            "context_features": torch.from_numpy(self.features[context_ids]),
            "context_ratings": torch.from_numpy(context_ratings),
            "context_mask": torch.from_numpy(context_mask),
            "cluster_features": torch.from_numpy(np.stack(cluster_features).astype(np.float32)),
            "target_features": torch.from_numpy(self.features[np.asarray(target_ids, dtype=np.int64)]),
            "target_owner": torch.tensor(owners, dtype=torch.long),
            "target_ratings": torch.tensor(target_ratings, dtype=torch.float32),
        }


class ClusterDenoisingAutoencoder(nn.Module):
    """Set encoder -> bottleneck -> item-conditioned rating-class decoder."""

    def __init__(self, embedding_dim=768, feature_dim=64, latent_dim=32,
                 num_rating_classes=5):
        super().__init__()
        if num_rating_classes < 2:
            raise ValueError("num_rating_classes must be at least 2.")
        self.num_rating_classes = num_rating_classes
        self.register_buffer(
            "scaled_rating_values", torch.linspace(0.0, 1.0, num_rating_classes)
        )
        self.item_encoder = nn.Sequential(nn.Linear(embedding_dim, 128), nn.ReLU(), nn.Linear(128, feature_dim))
        self.rating_encoder = nn.Sequential(nn.Linear(feature_dim + 1, 128), nn.ReLU(), nn.Linear(128, feature_dim))
        self.cluster_encoder = nn.Sequential(nn.Linear(embedding_dim, feature_dim), nn.ReLU())
        self.encoder = nn.Sequential(nn.Linear(2 * feature_dim + 1, 128), nn.ReLU(), nn.Linear(128, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + 2 * feature_dim, 128), nn.ReLU(),
            nn.Linear(128, num_rating_classes),
        )

    def encode(self, context_features, context_ratings, context_mask, cluster_features):
        item_codes = self.item_encoder(context_features)
        observations = self.rating_encoder(torch.cat([item_codes, context_ratings.unsqueeze(-1)], dim=-1))
        mask = context_mask.unsqueeze(-1).to(observations.dtype)
        counts = mask.sum(dim=1)
        pooled = (observations * mask).sum(dim=1) / counts.clamp(min=1)
        cluster_codes = self.cluster_encoder(cluster_features)
        latent = self.encoder(torch.cat([pooled, cluster_codes, counts.log1p()], dim=-1))
        return latent, cluster_codes

    def decode_logits(self, latent, cluster_codes, target_features, target_owner):
        target_codes = self.item_encoder(target_features)
        inputs = torch.cat([latent[target_owner], target_codes, cluster_codes[target_owner]], dim=-1)
        return self.decoder(inputs)

    def expected_scaled_rating(self, logits):
        return (logits.softmax(dim=-1) * self.scaled_rating_values).sum(dim=-1)

    def decode(self, latent, cluster_codes, target_features, target_owner):
        logits = self.decode_logits(latent, cluster_codes, target_features, target_owner)
        return self.expected_scaled_rating(logits)

    def forward_logits(self, batch):
        latent, cluster_codes = self.encode(batch["context_features"], batch["context_ratings"],
                                            batch["context_mask"], batch["cluster_features"])
        return self.decode_logits(latent, cluster_codes, batch["target_features"],
                                  batch["target_owner"])

    def forward(self, batch):
        latent, cluster_codes = self.encode(batch["context_features"], batch["context_ratings"],
                                            batch["context_mask"], batch["cluster_features"])
        return self.decode(latent, cluster_codes, batch["target_features"], batch["target_owner"])


def _scaled_ratings_to_classes(targets, num_classes):
    positions = targets * (num_classes - 1)
    classes = positions.round().long()
    if not torch.allclose(positions, classes.to(positions.dtype), atol=1e-4, rtol=0):
        raise ValueError("Focal rating loss requires discrete, evenly spaced rating classes.")
    if classes.numel() and (classes.min() < 0 or classes.max() >= num_classes):
        raise ValueError("A target lies outside the configured rating classes.")
    return classes


def build_rating_class_weights(training_groups, num_classes=5, class_weight="balanced"):
    """Rating-class weights computed only from training targets.

    "balanced" matches scikit-learn: n_samples / (n_classes * count), so every rating
    class carries the same total weight. None gives every class weight 1.
    """
    if num_classes < 2:
        raise ValueError("Use at least two rating classes.")
    if class_weight not in ("balanced", None):
        raise ValueError('class_weight must be "balanced" or None.')
    values = np.concatenate([group.ratings for group in training_groups])
    positions = values * (num_classes - 1)
    classes = np.rint(positions).astype(np.int64)
    if not np.allclose(positions, classes, atol=1e-4, rtol=0):
        raise ValueError("Class weights require discrete, evenly spaced rating classes.")
    counts = np.bincount(classes, minlength=num_classes)
    if class_weight is None:
        return counts, np.ones(num_classes, dtype=np.float32)
    present = np.flatnonzero(counts)
    weights = np.zeros(num_classes, dtype=np.float32)
    weights[present] = compute_class_weight("balanced", classes=present, y=classes)
    # A rating absent from training cannot be fit; weigh its validation rows like the rarest rating.
    weights[counts == 0] = weights.max()
    return counts, weights


def focal_rating_loss(logits, targets, class_weights, gamma=2.0):
    """Class-weighted multiclass focal loss for discrete ratings."""
    if gamma < 0:
        raise ValueError("Focal gamma must be nonnegative.")
    classes = _scaled_ratings_to_classes(targets, logits.shape[-1])
    log_probabilities = torch.log_softmax(logits, dim=-1)
    target_log_probability = log_probabilities.gather(1, classes.unsqueeze(1)).squeeze(1)
    target_probability = target_log_probability.exp()
    alpha = class_weights[classes]
    return (alpha * (1 - target_probability).pow(gamma) * -target_log_probability).mean()


def evaluate_autoencoder(model, loader, device, rating_span, class_weights=None,
                         focal_gamma=2.0):
    model.eval()
    squared_error = absolute_error = focal_total = 0.0
    count = 0
    class_squared = np.zeros(model.num_rating_classes, dtype=np.float64)
    class_counts = np.zeros(model.num_rating_classes, dtype=np.int64)
    weights = (torch.as_tensor(class_weights, dtype=torch.float32, device=device)
               if class_weights is not None else None)
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model.forward_logits(batch)
            predictions = model.expected_scaled_rating(logits)
            errors = (predictions - batch["target_ratings"]) * rating_span
            squared_error += errors.square().sum().item()
            absolute_error += errors.abs().sum().item()
            count += errors.numel()
            classes = _scaled_ratings_to_classes(
                batch["target_ratings"], model.num_rating_classes
            )
            for class_id in range(model.num_rating_classes):
                selected = classes == class_id
                if selected.any():
                    class_squared[class_id] += errors[selected].square().sum().item()
                    class_counts[class_id] += selected.sum().item()
            if weights is not None:
                focal_total += focal_rating_loss(
                    logits, batch["target_ratings"], weights, gamma=focal_gamma
                ).item() * errors.numel()
    present = class_counts > 0
    class_rmse = np.full(model.num_rating_classes, np.nan, dtype=np.float64)
    class_rmse[present] = np.sqrt(class_squared[present] / class_counts[present])
    return {
        "rmse": (squared_error / count) ** 0.5 if count else np.nan,
        "mae": absolute_error / count if count else np.nan,
        "macro_rmse": np.nanmean(class_rmse) if present.any() else np.nan,
        "focal_loss": focal_total / count if count and weights is not None else np.nan,
        "class_rmse": class_rmse, "class_counts": class_counts, "count": count,
    }


def train_autoencoder(model, training_groups, validation_pairs, collator, device,
                      epochs=10, batch_size=64, learning_rate=1e-3, mask_fraction=0.3,
                      max_context=64, max_targets=32, rating_span=4.0,
                      class_weights=None, focal_gamma=2.0, seed=42):
    if epochs < 1 or focal_gamma < 0:
        raise ValueError("Epochs must be positive and focal_gamma must be nonnegative.")
    model.to(device)
    if class_weights is None:
        _, class_weights = build_rating_class_weights(
            training_groups, num_classes=model.num_rating_classes
        )
    class_weights = torch.as_tensor(class_weights, dtype=torch.float32, device=device)
    if class_weights.shape != (model.num_rating_classes,) or not torch.isfinite(class_weights).all():
        raise ValueError("class_weights must contain one finite value per rating class.")
    train_data = MaskedRatingDataset(training_groups, mask_fraction, max_context, max_targets, seed)
    val_data = ValidationRatingDataset(validation_pairs, max_context, seed)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collator,
                        num_workers=0, generator=generator)
    validation_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=collator,
                                   num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_state, best_focal_loss = [], None, np.inf
    for epoch in range(1, epochs + 1):
        model.train()
        squared_error = focal_loss_total = 0.0
        count = 0
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            logits = model.forward_logits(batch)
            loss = focal_rating_loss(
                logits, batch["target_ratings"], class_weights, gamma=focal_gamma
            )  # Only masked, truly observed targets.
            loss.backward()
            optimizer.step()
            errors = model.expected_scaled_rating(logits.detach()) - batch["target_ratings"]
            focal_loss_total += loss.detach().item() * errors.numel()
            squared_error += errors.detach().square().sum().item() * rating_span ** 2
            count += errors.numel()
        metrics = evaluate_autoencoder(
            model, validation_loader, device, rating_span, class_weights, focal_gamma
        )
        history.append({"epoch": epoch, "train_focal_loss": focal_loss_total / count,
                        "train_masked_rmse": (squared_error / count) ** 0.5,
                        "validation_focal_loss": metrics["focal_loss"],
                        "validation_rmse": metrics["rmse"],
                        "validation_macro_rmse": metrics["macro_rmse"],
                        "validation_mae": metrics["mae"],
                        "validation_ratings": metrics["count"]})
        print(history[-1], flush=True)
        if metrics["count"] and metrics["focal_loss"] < best_focal_loss:
            best_focal_loss = metrics["focal_loss"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return pd.DataFrame(history)


def validation_baselines(training_groups, validation_pairs, rating_span=4.0):
    """Global and user/cluster means fitted exclusively on training observations."""
    total = sum(float(group.ratings.sum()) for group in training_groups)
    count = sum(len(group.ratings) for group in training_groups)
    global_mean = total / count
    errors = {"global_mean": [], "user_cluster_mean": []}
    for context, targets in validation_pairs:
        errors["global_mean"].extend((global_mean - targets.ratings) * rating_span)
        errors["user_cluster_mean"].extend((context.ratings.mean() - targets.ratings) * rating_span)
    rows = []
    for name, values in errors.items():
        values = np.asarray(values)
        rows.append({"baseline": name, "rmse": np.sqrt(np.mean(values ** 2)) if len(values) else np.nan,
                     "mae": np.mean(np.abs(values)) if len(values) else np.nan, "count": len(values)})
    return pd.DataFrame(rows).set_index("baseline")


def predict_unrated_items(model, user_id, catalog, observed_ratings, feature_matrix,
                          cluster_embeddings, device, rating_min=1.0, rating_max=5.0,
                          candidate_sources=("recipe", "food"), batch_size=1024,
                          max_context=64, top_k=None, seed=42):
    """Score unrated items only in clusters where this user has an observed rating.

    Use training-only observations for evaluation. At serving time it is valid to
    use all known observations, including the ratings held out during validation.
    A user without a non-noise rated cluster receives an empty result.
    """
    if batch_size < 1 or max_context < 1 or (top_k is not None and top_k < 1):
        raise ValueError("Batch size, context limit, and optional top_k must be positive.")
    user_rows = observed_ratings.loc[observed_ratings["user_id"].astype(str) == str(user_id)]
    rated = set(user_rows["item_index"])
    eligible = user_rows.loc[user_rows["cluster"] >= 0]
    output_columns = ["user_id", "item_key", "source", "title", "cluster", "predicted_rating", "context_ratings"]
    results = []
    model.eval()
    embedding_ids = catalog["embedding_row_id"].to_numpy(dtype=np.int64)
    rng = np.random.default_rng(seed)
    with torch.inference_mode():
        for cluster, history in eligible.groupby("cluster", sort=True):
            candidates = catalog.loc[(catalog["cluster"] == cluster) & catalog["source"].isin(candidate_sources)
                                     & ~catalog.index.isin(rated)]
            if candidates.empty:
                continue
            chosen = rng.permutation(len(history))[:max_context]
            context = history.iloc[chosen]
            context_ids = context["item_index"].to_numpy(dtype=np.int64)
            context_features = torch.from_numpy(feature_matrix[embedding_ids[context_ids]][None]).to(device)
            values = torch.tensor(context["scaled_rating"].to_numpy()[None], dtype=torch.float32, device=device)
            mask = torch.ones(values.shape, dtype=torch.bool, device=device)
            centroid = torch.tensor(cluster_embeddings.loc[cluster, "average_embedding"][None],
                                    dtype=torch.float32, device=device)
            latent, cluster_codes = model.encode(context_features, values, mask, centroid)
            for start in range(0, len(candidates), batch_size):
                selected = candidates.iloc[start:start + batch_size]
                target_features = torch.from_numpy(feature_matrix[embedding_ids[selected.index.to_numpy()]])
                owners = torch.zeros(len(selected), dtype=torch.long, device=device)
                predictions = model.decode(latent, cluster_codes, target_features.to(device), owners)
                output = selected[["item_key", "source", "title", "cluster"]].copy()
                output["user_id"] = str(user_id)
                output["predicted_rating"] = rating_min + predictions.cpu().numpy() * (rating_max - rating_min)
                output["context_ratings"] = len(context)
                results.append(output[output_columns])
    if not results:
        return pd.DataFrame(columns=output_columns)
    result = pd.concat(results, ignore_index=True).sort_values("predicted_rating", ascending=False, kind="stable")
    return result.head(top_k).reset_index(drop=True) if top_k is not None else result.reset_index(drop=True)

In [ ]:
DATA_DIR = Path("input_stage2")
dataset_paths = {
    "new_dataset_recommendation": DATA_DIR / "recipe_user_ratings.csv",
    "current_dataset": DATA_DIR / "price_mapped_nutrients.csv",
}
missing_paths = [str(path) for path in dataset_paths.values() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Restore the missing CSVs or fix dataset_paths before running: "
        + ", ".join(missing_paths)
        + ". Broken symlinks also count as missing files."
    )

new_dataset_recommendation_df = pd.read_csv(dataset_paths["new_dataset_recommendation"])
current_dataset_df = pd.read_csv(dataset_paths["current_dataset"])

# Use 0 as the explicit preprocessing value for items with no observed rating.
# ZERO_IS_MISSING below keeps these rows out of supervised rating targets.
new_dataset_recommendation_df["rating"] = (
    pd.to_numeric(new_dataset_recommendation_df["rating"], errors="coerce").fillna(0.0)
)
current_dataset_df["rating"] = (
    pd.to_numeric(current_dataset_df.get("rating", 0.0), errors="coerce").fillna(0.0)
    if "rating" in current_dataset_df
    else 0.0
)

In [ ]:
display(new_dataset_recommendation_df.head())
display(current_dataset_df.head())

**Rating distribution**

The left chart shows each observed rating as a percentage of all rated
user–recipe observations. The right chart includes every observation and treats
rating `0` as unrated, matching the recommender's `ZERO_IS_MISSING` policy.

In [ ]:
import matplotlib.pyplot as plt

rating_values = pd.to_numeric(
    new_dataset_recommendation_df["rating"], errors="coerce"
).fillna(0.0)
rated_mask = rating_values.ne(0)

# Percentages for ratings 1–5 use rated observations as the denominator.
rating_levels = pd.Index(range(1, 6), name="rating")
rating_counts = rating_values.loc[rated_mask].value_counts().reindex(rating_levels, fill_value=0)
rating_percentages = rating_counts.div(rating_counts.sum()).mul(100)

rated_status_counts = pd.Series({
    "Rated": int(rated_mask.sum()),
    "Unrated": int((~rated_mask).sum()),
})
rated_status_percentages = rated_status_counts.div(rated_status_counts.sum()).mul(100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(
    rating_percentages.index.astype(str), rating_percentages.values,
    color=plt.cm.viridis(np.linspace(0.15, 0.9, len(rating_percentages))),
)
axes[0].bar_label(bars, labels=[f"{value:.1f}%" for value in rating_percentages], padding=3)
axes[0].set(title="Percentage of each observed rating", xlabel="Rating", ylabel="Rated observations (%)")
axes[0].set_ylim(0, max(5, rating_percentages.max() * 1.15))
axes[0].grid(axis="y", alpha=0.25)

wedges, _, _ = axes[1].pie(
    rated_status_percentages.values, labels=rated_status_percentages.index,
    autopct="%1.1f%%", startangle=90, colors=["#2a9d8f", "#e9c46a"],
    wedgeprops={"width": 0.45, "edgecolor": "white"},
)
axes[1].set_title("Rated vs unrated observations")
axes[1].legend(
    wedges, [f"{label}: {count:,}" for label, count in rated_status_counts.items()],
    loc="lower center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False,
)
plt.tight_layout()
plt.show()

display(pd.DataFrame({"count": rating_counts, "percentage_of_rated": rating_percentages.round(2)}))
display(pd.DataFrame({"count": rated_status_counts, "percentage": rated_status_percentages.round(2)}))

In [ ]:
MODEL_NAME = "bert-base-uncased"
FEATURE_COLUMNS = ("products", "adj", "verb")
COLUMN_WEIGHTS = {"products": 0.6, "adj": 0.2, "verb": 0.2}
BERT_BATCH_SIZE = 32
MAX_LENGTH = 512  # Longer individual phrases are truncated at BERT's token limit.

weights = np.array([COLUMN_WEIGHTS[column] for column in FEATURE_COLUMNS])
if not (
    np.isfinite(weights).all()
    and weights[0] > weights[1] > 0
    and weights[1] == weights[2]
    and np.isclose(weights.sum(), 1.0)
):
    raise ValueError("Use positive weights summing to 1 with products > adj = verb.")

RANDOM_STATE = 42
UMAP_COMPONENTS = 10
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST = 0.0

# Content-only attention pretraining; these are separate from rating AE settings.
ENCODER_HEADS = 8
ENCODER_FEEDFORWARD_DIM = 1536
ENCODER_DROPOUT = 0.1
ENCODER_EPOCHS = 10
ENCODER_BATCH_SIZE = 64
ENCODER_LEARNING_RATE = 1e-4
ENCODER_VALIDATION_FRACTION = 0.1

**Preserve individual terms and share repeated base embeddings**

Nested lists such as `[["olive oil"], ["salt", "salt"]]` become three phrases:
`olive oil`, `salt`, `salt`. Multiword ingredients and repeated occurrences remain
intact; `[[], []]` is an empty field. Sort terms within each field to canonicalize
feature combinations without imposing a sequence order on attention.

Complete combinations share one feature ID across both datasets. Original row
order, duplicate indexes, and row counts are preserved. Pretraining visits each
unique combination once per epoch; rating frequency does not multiply its weight.
Only frozen BERT phrase vectors are cached globally. Contextualized vectors are
specific to the complete recipe/food feature combination.

In [ ]:
datasets = {
    "new_dataset_recommendation": new_dataset_recommendation_df,
    "current_dataset": current_dataset_df,
}
term_feature_table = prepare_term_features(datasets)
unique_features = term_feature_table.unique_features
row_embedding_ids = term_feature_table.row_embedding_ids
row_counts = term_feature_table.row_counts
print(
    f"{len(row_embedding_ids):,} rows; {len(unique_features):,} unique feature "
    f"combinations; {len(term_feature_table.terms):,} distinct phrases."
)
display(unique_features.head())

**Create frozen BERT phrase embeddings**

Encode each distinct phrase once. Mean pooling excludes padding and special tokens,
and each phrase vector is unit-normalized. BERT runs in evaluation mode with all
parameters frozen. These cached vectors supply both attention inputs and fixed
masked-term reconstruction targets. Recipe-level pooling happens only after attention.

In [ ]:
# Prefer NVIDIA CUDA, then Apple Silicon's PyTorch MPS backend.
# Probe the selected backend so training never silently falls back to CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    raise RuntimeError(
        "No supported accelerator found. Use an NVIDIA CUDA runtime or Apple Silicon with MPS-enabled PyTorch."
    )
try:
    _accelerator_probe = torch.ones(1, device=device) + 1
    if device.type == "cuda":
        torch.cuda.synchronize()
    del _accelerator_probe
except RuntimeError as exc:
    raise RuntimeError(f"{device.type.upper()} was detected but cannot execute in this runtime.") from exc
torch.set_float32_matmul_precision("high")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval().requires_grad_(False)
print(f"Embedding individual phrases with frozen {MODEL_NAME} on {device}.")
term_embeddings = embed_terms(
    term_feature_table.terms, tokenizer, bert_model, device,
    batch_size=BERT_BATCH_SIZE, max_length=MAX_LENGTH,
)
# Attention learns from the cached NumPy vectors, without retaining BERT on the GPU.
bert_model.to("cpu")
if device.type == "cuda":
    torch.cuda.empty_cache()
print(f"Cached phrase embedding matrix: {term_embeddings.shape}")

**Pretrain recipe attention, then freeze it**

```text
Ingredient BERT phrases ── self-attention ── queries ───────────────┐
Adjective BERT phrases  ── self-attention ── keys/values ───────────┤
Verb BERT phrases       ── self-attention ── keys/values ───────────┤
                                                                 ▼
                                             Ingredient cross-attention
                                                                 │
                       Contextualized ingredients + adjectives + verbs
                                                                 ▼
                        Masked mean pooling per field + normalization
                                                                 ▼
                              0.6 products + 0.2 adj + 0.2 verb
                                                                 ▼
                              UMAP → HDBSCAN → rating autoencoder
```

For each eligible unique combination, hide one random term behind a learned mask
vector before any attention. Field-specific heads reconstruct its original BERT
vector using cosine loss. Ingredients can use context from any field; adjectives
and verbs need another term in their own field to be eligible targets. Examples
without eligible targets still receive embeddings during inference.

Hold out 10% of eligible **complete feature combinations**, with fixed validation
masks. Restore the best validation checkpoint; if only one combination is eligible,
there is no holdout and the final epoch is retained. BERT stays frozen throughout.
No ratings are used. This is content pretraining over the loaded catalog, including
items whose ratings may later be held out for recommender validation.

After pretraining, freeze all encoder parameters. Empty attention groups are
bypassed and padding never contributes to pooling. The attention encoder stays
frozen during clustering and rating training so cluster features remain consistent.

In [ ]:
torch.manual_seed(RANDOM_STATE)
recipe_encoder = RecipeContextEncoder(
    embedding_dim=term_embeddings.shape[1], num_heads=ENCODER_HEADS,
    feedforward_dim=ENCODER_FEEDFORWARD_DIM, dropout=ENCODER_DROPOUT,
    column_weights=COLUMN_WEIGHTS,
)
encoder_training_config = {
    "epochs": ENCODER_EPOCHS,
    "batch_size": ENCODER_BATCH_SIZE,
    "learning_rate": ENCODER_LEARNING_RATE,
    "validation_fraction": ENCODER_VALIDATION_FRACTION,
    "seed": RANDOM_STATE,
}
encoder_training_history = train_recipe_encoder(
    recipe_encoder, term_embeddings, term_feature_table.term_ids, device,
    **encoder_training_config,
)
display(encoder_training_history)
eligible_features = int(encoder_training_history.iloc[0]["training_features"]
                        + encoder_training_history.iloc[0]["validation_features"])
print(f"Pretrained on {eligible_features:,} eligible combinations; "
      f"{len(unique_features) - eligible_features:,} combinations are inference-only.")
assert not recipe_encoder.training
assert not any(parameter.requires_grad for parameter in recipe_encoder.parameters())

# Save immediately so the encoder checkpoint survives later clustering diagnostics.
MODEL_OUTPUT_DIR = Path("model_artifacts")
ENCODER_WEIGHTS_PATH = MODEL_OUTPUT_DIR / "recipe_context_encoder.h5"
ENCODER_IDENTITY = save_recipe_encoder(
    recipe_encoder, ENCODER_WEIGHTS_PATH, MODEL_NAME, MAX_LENGTH, encoder_training_config,
)
print(f"Saved frozen encoder: {ENCODER_WEIGHTS_PATH.resolve()}")
print(f"Encoder identity: {ENCODER_IDENTITY}")

In [ ]:
contextual_field_embeddings, contextualized_embeddings = encode_feature_table(
    recipe_encoder, term_embeddings, term_feature_table.term_ids, device,
    batch_size=ENCODER_BATCH_SIZE,
)
weighted_embeddings = contextualized_embeddings  # Compatibility alias, same matrix.

# Share pooled vectors by complete feature ID; never persist all term outputs.
field_embedding_rows = {
    field: pd.Series(list(matrix), dtype=object)
    for field, matrix in contextual_field_embeddings.items()
}
contextualized_embedding_rows = pd.Series(list(contextualized_embeddings), dtype=object)
offset = 0
for df in datasets.values():
    ids = row_embedding_ids[offset:offset + len(df)]
    df["embedding_row_id"] = ids
    for field in FEATURE_COLUMNS:
        df[f"{field}_embedding"] = field_embedding_rows[field].iloc[ids].to_numpy()
    df["contextualized_embedding"] = contextualized_embedding_rows.iloc[ids].to_numpy()
    df["weighted_embedding"] = df["contextualized_embedding"]  # Compatibility column.
    offset += len(df)
print(f"Contextualized recipe/food embedding matrix: {contextualized_embeddings.shape}")

**Reduce dimensions with UMAP, then cluster both datasets with HDBSCAN**

Fit one UMAP model on nonempty, unique contextualized vectors from both datasets.
Use cosine input distance, 10 output dimensions, 30 neighbors, and `min_dist=0.0`.
HDBSCAN uses **Euclidean distance in the shared reduced space**. The seed is fixed;
UMAP and HDBSCAN never use ratings or user IDs.

HDBSCAN fits each unique feature combination once, so repeated user-rating rows do
not alter content clusters. Original row counts are applied only when reporting
cluster sizes. Zero vectors remain noise (`-1`), and all valid unique vectors are
clustered without sampling.

Tune `MIN_CLUSTER_SIZE`, `MIN_SAMPLES`, and `CLUSTER_SELECTION_EPSILON` after
inspecting cluster sizes, noise, and membership probabilities. Attention changes
the geometry, so parameter choices from the earlier clustering method do not transfer directly.
Cluster means and recommender features use the frozen encoder's original
768-dimensional contextualized space, independently of UMAP coordinates.

In [ ]:
valid_embedding_ids = np.flatnonzero(np.linalg.norm(contextualized_embeddings, axis=1) > 0)
# NaN denotes a missing embedding; the coordinate zero itself is a valid UMAP point.
reduced_embeddings = np.full((len(contextualized_embeddings), UMAP_COMPONENTS), np.nan, dtype=np.float32)
umap_model = None
if len(valid_embedding_ids):
    if len(valid_embedding_ids) < 3:
        raise ValueError("UMAP needs at least three nonempty unique feature combinations.")
    umap_model = UMAP(
        n_components=UMAP_COMPONENTS,
        n_neighbors=min(UMAP_NEIGHBORS, len(valid_embedding_ids) - 1),
        min_dist=UMAP_MIN_DIST,
        metric="cosine",
        init="random",  # Also works when a small sample has fewer rows than output dimensions.
        random_state=RANDOM_STATE,
        n_jobs=1,
        low_memory=True,
    )
    reduced_embeddings[valid_embedding_ids] = umap_model.fit_transform(
        contextualized_embeddings[valid_embedding_ids]
    )
    if not np.isfinite(reduced_embeddings[valid_embedding_ids]).all():
        raise ValueError("UMAP produced non-finite coordinates. Inspect the inputs and neighbor settings.")

reduced_embedding_rows = pd.Series(list(reduced_embeddings), dtype=object)
for df in datasets.values():
    df["umap_embedding"] = reduced_embedding_rows.iloc[df["embedding_row_id"].to_numpy()].to_numpy()
print(f"UMAP dimensions: {contextualized_embeddings.shape[1]} → {UMAP_COMPONENTS}")

In [ ]:
# HDBSCAN parameters
MIN_CLUSTER_SIZE = 20
MIN_SAMPLES = 20
CLUSTER_SELECTION_EPSILON = 0.0
CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_N_JOBS = -1

In [ ]:
hdbscan_configuration = pd.Series({
    "min_cluster_size": MIN_CLUSTER_SIZE,
    "min_samples": MIN_SAMPLES,
    "cluster_selection_epsilon": CLUSTER_SELECTION_EPSILON,
    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
    "n_jobs": HDBSCAN_N_JOBS,
}, name="value")
display(hdbscan_configuration.to_frame())

In [ ]:
def cluster_contextualized_embeddings(
    embeddings, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES,
    cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
    cluster_selection_method=CLUSTER_SELECTION_METHOD, n_jobs=HDBSCAN_N_JOBS,
):
    if min_cluster_size < 2 or min_samples < 1:
        raise ValueError("min_cluster_size must be >= 2 and min_samples must be positive.")
    if not np.isfinite(cluster_selection_epsilon) or cluster_selection_epsilon < 0:
        raise ValueError("cluster_selection_epsilon must be finite and nonnegative.")
    if cluster_selection_method not in {"eom", "leaf"}:
        raise ValueError("cluster_selection_method must be 'eom' or 'leaf'.")
    labels = np.full(len(embeddings), -1, dtype=np.int64)
    probabilities = np.zeros(len(embeddings), dtype=np.float32)
    valid_ids = np.flatnonzero(np.isfinite(embeddings).all(axis=1))
    if not len(valid_ids):
        return labels, probabilities, None

    if len(valid_ids) < max(min_cluster_size, min_samples):
        return labels, probabilities, None
    estimator = HDBSCAN(
        min_cluster_size=min_cluster_size, min_samples=min_samples,
        cluster_selection_epsilon=cluster_selection_epsilon,
        cluster_selection_method=cluster_selection_method, metric="euclidean",
        algorithm="auto", n_jobs=n_jobs, allow_single_cluster=False,
    )
    labels[valid_ids] = estimator.fit_predict(embeddings[valid_ids])
    probabilities[valid_ids] = estimator.probabilities_.astype(np.float32, copy=False)
    return labels, probabilities, estimator


unique_cluster_labels, unique_cluster_probabilities, hdbscan_model = (
    cluster_contextualized_embeddings(reduced_embeddings)
)
for df in datasets.values():
    df["cluster"] = unique_cluster_labels[df["embedding_row_id"].to_numpy()]
    df["cluster_probability"] = unique_cluster_probabilities[df["embedding_row_id"].to_numpy()]

cluster_sizes = pd.Series(
    unique_cluster_labels[row_embedding_ids], name="cluster"
).value_counts().sort_index().rename("row_count")
print(f"Clusters: {(cluster_sizes.index >= 0).sum()}; noise rows: {cluster_sizes.get(-1, 0):,}")
display(cluster_sizes.to_frame())

**Cluster scatter plot**

Plot the first two dimensions of the shared UMAP projection. Each point is one
unique ingredient/adjective/verb combination; color identifies its HDBSCAN cluster,
point size reflects membership probability, and gray points are HDBSCAN noise (`-1`).
The plot uses all valid unique points.

In [ ]:
import matplotlib.pyplot as plt

if reduced_embeddings.shape[1] < 2:
    raise ValueError("The cluster scatter plot requires at least two UMAP dimensions.")

plot_mask = np.isfinite(reduced_embeddings[:, :2]).all(axis=1)
noise_mask = plot_mask & (unique_cluster_labels == -1)
clustered_mask = plot_mask & (unique_cluster_labels >= 0)

fig, ax = plt.subplots(figsize=(11, 8))
if noise_mask.any():
    ax.scatter(
        reduced_embeddings[noise_mask, 0], reduced_embeddings[noise_mask, 1],
        c="#b8b8b8", s=5, alpha=0.3, linewidths=0, label=f"Noise ({noise_mask.sum():,})",
        rasterized=True,
    )
if clustered_mask.any():
    cluster_points = ax.scatter(
        reduced_embeddings[clustered_mask, 0], reduced_embeddings[clustered_mask, 1],
        c=unique_cluster_labels[clustered_mask], cmap="turbo",
        s=4 + 12 * unique_cluster_probabilities[clustered_mask],
        alpha=0.65, linewidths=0, rasterized=True,
    )
    colorbar = fig.colorbar(cluster_points, ax=ax, pad=0.01)
    colorbar.set_label("HDBSCAN cluster ID")
if noise_mask.any():
    ax.legend(frameon=False, markerscale=2)
ax.set(
    title=(f"HDBSCAN clusters in shared UMAP space — "
           f"{np.unique(unique_cluster_labels[clustered_mask]).size} clusters"),
    xlabel="UMAP dimension 1", ylabel="UMAP dimension 2",
)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()

**Average contextualized embedding for each cluster**

For cluster C, `average_embedding = sum(row_count[i] * contextualized_embeddings[i])
/ sum(row_count[i])`, using its unique feature combinations i. This is exactly the
mean over all original member rows from both datasets. These means are not
normalized again; a separate normalized copy supports cosine comparisons.

In [ ]:
def average_cluster_embeddings(embeddings, labels, counts):
    valid_ids = np.flatnonzero(labels >= 0)
    members = valid_ids[np.argsort(labels[valid_ids], kind="stable")]
    cluster_ids, starts = np.unique(labels[members], return_index=True)
    means = np.empty((len(cluster_ids), embeddings.shape[1]), dtype=np.float32)
    sizes = np.zeros(len(cluster_ids), dtype=np.int64)
    for position, cluster_id in enumerate(cluster_ids):
        stop = starts[position + 1] if position + 1 < len(starts) else len(members)
        member_ids = members[starts[position]:stop]
        sizes[position] = counts[member_ids].sum()
        total = np.zeros(embeddings.shape[1], dtype=np.float64)
        for start in range(0, len(member_ids), 4096):
            ids = member_ids[start:start + 4096]
            total += np.einsum("i,ij->j", counts[ids], embeddings[ids], dtype=np.float64)
        means[position] = total / sizes[position]
    summary = pd.DataFrame({
        "cluster": cluster_ids,
        "row_count": sizes,
        "average_embedding": list(means),
    }).set_index("cluster")
    return summary, means


cluster_embeddings_df, cluster_embedding_matrix = average_cluster_embeddings(
    contextualized_embeddings, unique_cluster_labels, row_counts
)
# Matrix row order follows cluster_embeddings_df.index, including when there are no clusters.
cluster_embedding_matrix_normalized = (
    normalize(cluster_embedding_matrix, norm="l2")
    if len(cluster_embedding_matrix) else cluster_embedding_matrix.copy()
)
for dataset_name, df in datasets.items():
    source_counts = df.loc[df["cluster"] >= 0, "cluster"].value_counts()
    cluster_embeddings_df[f"{dataset_name}_rows"] = source_counts.reindex(
        cluster_embeddings_df.index, fill_value=0
    ).astype(np.int64)

display(cluster_embeddings_df.head())
for dataset_name, df in datasets.items():
    print(dataset_name)
    product_column = "products" if "products" in df.columns else "product"
    display(df[[product_column, "adj", "verb", "cluster"]].head())

Both original dataframes retain their contextualized `products_embedding`,
`adj_embedding`, `verb_embedding`, and combined `contextualized_embedding` columns,
plus `cluster`. Treat shared vector arrays as read-only. `weighted_embedding` and
`weighted_embeddings` remain compatibility aliases for the contextualized features.

`cluster_embeddings_df` contains one contextualized average per non-noise cluster;
`cluster_embedding_matrix` contains these averages as a numeric matrix.
For example, `cluster_embeddings_df.loc[cluster_id, "average_embedding"]` retrieves
its mean in the frozen encoder's feature space.

References: [BERT](https://huggingface.co/google-bert/bert-base-uncased),
[PyTorch multihead attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html),
[HDBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html),
and [UMAP for clustering](https://umap-learn.readthedocs.io/en/latest/clustering.html).

**Recommendation architecture: a cluster-conditioned denoising autoencoder**

The recipe attention encoder has already been pretrained and frozen. Its same
contextualized vectors supply visible items, targets, and cluster means. The rating
autoencoder trains its existing item projections and bottleneck without updating
the recipe encoder or invalidating cluster assignments.

For each `(user, cluster)`, randomly hide 30% of the observed ratings on
each training pass. Encode only the remaining `(item embedding, rating)`
pairs, then reconstruct the hidden ratings using the latent user/cluster
representation, the target item's embedding, and the cluster mean.

```text
Visible rated items in ONE user/cluster group
      │ frozen contextualized vectors + ratings + explicit observation mask
      ▼
Shared item projection (768 → 64) + rating encoder
      ▼
Masked mean pooling ─────── Cluster mean projection (768 → 64)
      └────────────────────┬───────────────────────────────┘
                           ▼
                 Encoder → latent vector (32)
                           │
            Target item vector + cluster projection
                           ▼
             Decoder → five rating-class logits
                           ▼
          Softmax expectation → continuous rating
```

This is a **set denoising autoencoder**: a shared network accepts different
cluster sizes without creating a dense user-by-item matrix. It has no
learned item-ID lookup, so food items without ratings can be scored from
content. One model shares parameters across clusters; each example's
context and candidates stay within its own cluster. Food predictions are
transfer from recipe ratings and have no labeled food-rating validation.

Only randomly hidden **observed** ratings contribute to class-balanced focal
cross-entropy. Balanced class weights (`CLASS_WEIGHT = "balanced"`, as in scikit-learn)
give rating `r` the weight `n_targets / (5 × count_r)`, so every rating class carries the
same total weight and mistakes on rare ratings cost more. The focal factor also
emphasizes difficult examples. Rating 5 is
also downsampled only for prolific users before train/validation splitting.
Unrated items are never assigned zero targets. Fixed validation ratings
are withheld before training and are excluded from every training input
and target. Validation is available for groups with at least three
ratings; at least two observations remain for masked training. Groups
with two ratings train without a validation holdout. Singletons can be
recommendation contexts but cannot train reconstruction from other items.
Noise `-1` is excluded, rather than treated as a cluster.

Training contexts are capped at 64 sampled ratings and masked targets
at 32 per group to bound memory. Validation reports overall and macro-per-class
RMSE plus MAE on the original rating scale. The best checkpoint uses focal
validation loss and metrics are compared with training-only mean baselines.
Validation covers eligible groups, not all users or unrated foods.
A user with no rated, non-noise cluster receives no personalized results.

In [ ]:
# Dataset policy: change RATING_MIN to 0 and ZERO_IS_MISSING to False if zero is a real rating.
ZERO_IS_MISSING = True
RATING_MIN, RATING_MAX = 1.0, 5.0
CANDIDATE_SOURCES = ("recipe", "food")
VALIDATION_FRACTION = 0.2
MASK_FRACTION = 0.3
LATENT_DIM = 32
AE_EPOCHS = 10
AE_BATCH_SIZE = 64
AE_LEARNING_RATE = 1e-3
MAX_CONTEXT_RATINGS = 64
MAX_MASKED_TARGETS = 32
FOCAL_GAMMA = 2.0
CLASS_WEIGHT = "balanced"  # scikit-learn style; None gives every rating weight 1.
DOMINANT_RATING = 5.0
HEAVY_USER_MIN_RATINGS = 50
MAX_DOMINANT_TO_OTHER_RATIO = 2.0
MIN_DOMINANT_KEEP = 10
NUM_RATING_CLASSES = int(round(RATING_MAX - RATING_MIN)) + 1

torch.manual_seed(RANDOM_STATE)
balanced_recipe_ratings_df, rating_balance_summary = downsample_dominant_user_ratings(
    new_dataset_recommendation_df, dominant_rating=DOMINANT_RATING,
    heavy_user_min_ratings=HEAVY_USER_MIN_RATINGS,
    max_dominant_to_other_ratio=MAX_DOMINANT_TO_OTHER_RATIO,
    min_dominant_keep=MIN_DOMINANT_KEEP, zero_is_missing=ZERO_IS_MISSING,
    seed=RANDOM_STATE,
)
print(
    f"Downsampled {rating_balance_summary.attrs['removed_rows']:,} rating-{DOMINANT_RATING:g} "
    f"rows from {rating_balance_summary.attrs['heavy_users']:,} prolific users."
)
display(rating_balance_summary.round(2))

item_catalog, observed_ratings = build_catalog_and_ratings(
    new_dataset_recommendation_df, current_dataset_df,
    rating_min=RATING_MIN, rating_max=RATING_MAX, zero_is_missing=ZERO_IS_MISSING,
    rating_df=balanced_recipe_ratings_df,
)
serving_catalog, serving_observed_ratings = build_catalog_and_ratings(
    new_dataset_recommendation_df, current_dataset_df,
    rating_min=RATING_MIN, rating_max=RATING_MAX, zero_is_missing=ZERO_IS_MISSING,
)
if not item_catalog[["item_key"]].equals(serving_catalog[["item_key"]]):
    raise RuntimeError("Training and serving catalogs do not have identical item indexes.")
del serving_catalog
training_groups, validation_pairs = split_rating_groups(
    observed_ratings, validation_fraction=VALIDATION_FRACTION, seed=RANDOM_STATE
)
rating_class_counts, rating_class_weights = build_rating_class_weights(
    training_groups, num_classes=NUM_RATING_CLASSES, class_weight=CLASS_WEIGHT
)
rating_weight_table = pd.DataFrame({
    "rating": np.linspace(RATING_MIN, RATING_MAX, NUM_RATING_CLASSES),
    "training_targets": rating_class_counts,
    "focal_class_weight": rating_class_weights,
}).set_index("rating")
display(rating_weight_table.round(4))
clustered_ratings = observed_ratings.loc[observed_ratings["cluster"] >= 0]
serving_clustered_ratings = serving_observed_ratings.loc[
    serving_observed_ratings["cluster"] >= 0
]
coverage = pd.Series({
    "catalog_items": len(item_catalog),
    "full_observed_ratings": len(serving_observed_ratings),
    "balanced_observed_ratings": len(observed_ratings),
    "downsampled_rating_5_rows": rating_balance_summary.attrs["removed_rows"],
    "ratings_in_non_noise_clusters": len(clustered_ratings),
    "users_with_observed_ratings": observed_ratings["user_id"].nunique(),
    "users_with_rated_clusters": clustered_ratings["user_id"].nunique(),
    "training_user_cluster_groups": len(training_groups),
    "training_ratings": sum(len(group.items) for group in training_groups),
    "validation_ratings": sum(len(target.items) for _, target in validation_pairs),
}, name="count")
display(coverage.to_frame())
if not validation_pairs:
    print("No groups support held-out validation; training will retain the final epoch.")

In [ ]:
rating_collator = RatingCollator(contextualized_embeddings, item_catalog, cluster_embeddings_df)
recommendation_model = ClusterDenoisingAutoencoder(
    embedding_dim=contextualized_embeddings.shape[1], latent_dim=LATENT_DIM,
    num_rating_classes=NUM_RATING_CLASSES,
)
print(recommendation_model)
training_history = train_autoencoder(
    recommendation_model, training_groups, validation_pairs, rating_collator, device,
    epochs=AE_EPOCHS, batch_size=AE_BATCH_SIZE, learning_rate=AE_LEARNING_RATE,
    mask_fraction=MASK_FRACTION, max_context=MAX_CONTEXT_RATINGS,
    max_targets=MAX_MASKED_TARGETS, rating_span=RATING_MAX - RATING_MIN,
    class_weights=rating_class_weights, focal_gamma=FOCAL_GAMMA, seed=RANDOM_STATE,
)
baseline_metrics = validation_baselines(
    training_groups, validation_pairs, rating_span=RATING_MAX - RATING_MIN
)
display(training_history)
display(baseline_metrics)
# The lowest focal-validation-loss epoch is restored. If validation is empty, retain the final epoch.

**Predict the unrated items in a user's clusters**

At recommendation time, use all the user's genuinely known ratings as
context, including ratings previously set aside to measure validation.
This serving step is separate from validation. Predict every unrated
candidate from the configured sources in the user's rated clusters,
excluding already-rated items. Candidates with no ratings of their own
are supported through their frozen contextualized features. Scores are estimated
ratings; no value is written back as an observed training label.

In [ ]:
# Choose an existing user for the example; replace with any user ID to personalize.
example_user_id = serving_clustered_ratings["user_id"].value_counts().index[0]
predicted_unrated_ratings = predict_unrated_items(
    recommendation_model, example_user_id, item_catalog, serving_observed_ratings,
    contextualized_embeddings, cluster_embeddings_df, device,
    rating_min=RATING_MIN, rating_max=RATING_MAX,
    candidate_sources=CANDIDATE_SOURCES, max_context=MAX_CONTEXT_RATINGS,
    top_k=None, seed=RANDOM_STATE,
)
recommendations = predicted_unrated_ratings.head(20)
print(f"Predicted {len(predicted_unrated_ratings):,} unrated items for user {example_user_id}.")
display(recommendations)

The **Model definitions** cell contains recipe parsing, attention, content pretraining,
checkpoint loading, rating reconstruction, and prediction together. The notebook
runs without either local Python module. `recipe_encoder` is frozen;
`recommendation_model` is the trained rating autoencoder. Metrics are in `encoder_training_history`,
`training_history`, and `baseline_metrics`.

`predicted_unrated_ratings` contains the example user's candidate scores;
`recommendations` shows the top 20. Change `example_user_id` or call
`predict_unrated_items` for another user. All loaded rows are represented; content
pretraining uses unique feature combinations rather than repeated rating rows.

Save both model artifacts as a pair. `load_recipe_encoder(path, device)` restores
a frozen encoder and its metadata; check its `encoder_identity` against the
recommender HDF5 attribute before using them together. Changing encoder weights,
BERT model, phrase token limit, or pooling configuration requires regenerating
embeddings and clusters and retraining the recommender.

In [ ]:
# Save recommender weights paired with the encoder that generated these features.
if encoder_identity(recipe_encoder, MODEL_NAME, MAX_LENGTH) != ENCODER_IDENTITY:
    raise RuntimeError("Encoder changed after feature generation; rerun embedding, clustering, and rating training.")
import h5py

MODEL_OUTPUT_DIR = Path("model_artifacts")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_WEIGHTS_PATH = MODEL_OUTPUT_DIR / "recipe_recommender_autoencoder.h5"

state_dict = recommendation_model.state_dict()
with h5py.File(MODEL_WEIGHTS_PATH, "w") as h5_file:
    h5_file.attrs["format"] = "pytorch_state_dict"
    h5_file.attrs["model_class"] = recommendation_model.__class__.__name__
    h5_file.attrs["feature_format_version"] = FEATURE_FORMAT_VERSION
    h5_file.attrs["encoder_identity"] = ENCODER_IDENTITY
    h5_file.attrs["encoder_artifact"] = ENCODER_WEIGHTS_PATH.name
    h5_file.attrs["feature_dim"] = 64
    h5_file.attrs["embedding_dim"] = contextualized_embeddings.shape[1]
    h5_file.attrs["latent_dim"] = LATENT_DIM
    h5_file.attrs["num_rating_classes"] = NUM_RATING_CLASSES
    h5_file.attrs["rating_min"] = RATING_MIN
    h5_file.attrs["rating_max"] = RATING_MAX
    h5_file.attrs["loss"] = "class_balanced_focal_cross_entropy"
    h5_file.attrs["focal_gamma"] = FOCAL_GAMMA
    h5_file.attrs["class_weight"] = str(CLASS_WEIGHT)
    h5_file.attrs["rating_class_weights"] = rating_class_weights
    h5_file.attrs["dominant_rating"] = DOMINANT_RATING
    h5_file.attrs["heavy_user_min_ratings"] = HEAVY_USER_MIN_RATINGS
    h5_file.attrs["max_dominant_to_other_ratio"] = MAX_DOMINANT_TO_OTHER_RATIO
    h5_file.attrs["min_dominant_keep"] = MIN_DOMINANT_KEEP
    h5_file.attrs["clustering_algorithm"] = "HDBSCAN"
    h5_file.attrs["min_cluster_size"] = MIN_CLUSTER_SIZE
    h5_file.attrs["min_samples"] = MIN_SAMPLES
    h5_file.attrs["cluster_selection_epsilon"] = CLUSTER_SELECTION_EPSILON
    h5_file.attrs["cluster_selection_method"] = CLUSTER_SELECTION_METHOD
    weights_group = h5_file.create_group("state_dict")
    for name, tensor in state_dict.items():
        weights_group.create_dataset(
            name, data=tensor.detach().cpu().numpy(), compression="gzip"
        )

print(f"Saved {len(state_dict)} tensors to {MODEL_WEIGHTS_PATH.resolve()}")